# Validation Results Explained, and Which Model to Choose

## Natural-language request

> In another notebook, reason through the results of the validation set. Explain them in detail so that I can answer any question anyone asks, and say which model I should choose.

> **AI-assisted.** The analysis and explanations were drafted with an AI assistant from the outputs of notebooks 11 and 12 and the extra checks below. Read them critically and defend the choice in your own words. The final cell checks that every number quoted here still matches the data.

## How to use this notebook

1. Run the cells in order. Every claim in the text has a cell above or below it that produces the number.
2. **Part 1** is the short answer: read it first.
3. **Parts 2 to 8** take the results one at a time. Each has the **evidence**, **what it means**, and **challenges**: questions a reviewer is likely to ask, with answers.
4. **Part 9** is the recommendation and what you may and may not claim. **Part 10** holds the hardest cross-cutting questions. **Part 11** lists what would change the recommendation.

**Test is never opened.** Everything here uses train and val (the contract's choose period) only.

## Part 1. The short answer

**Choose CatBoost, with a separate buffer for each hotel (City Hotel 6 rooms, Resort Hotel 3 rooms), and compare it with the flat rule given its own per-hotel buffers (City Hotel 20, Resort Hotel 4).** XGBoost is an equally defensible alternative: the two are practically tied.

But say clearly what the validation set shows, because it is less flattering than the headline:

1. **Notebook 12's headline was a 16% cost reduction against the flat rule. Most of that came from how the buffer was set, not from better forecasts.** With one shared buffer for both hotels, the flat rule needs 16 rooms of safety to protect the City Hotel, and that is far too cautious at the Resort. Give every method its own buffer per hotel and the flat rule improves by €464,000. The best models are then only **1.0% to 1.4%** cheaper.
2. **That small edge is not reliable.** Resampling the val weeks, CatBoost is cheaper than the flat rule in **74%** of resamples. The best models win in only **3 of the 6** val months.
3. **All four models predict too few cancellations on val** (about 28% against an actual 36.9%). The main cause is one input, `previous_cancellations`, which carries a strong signal in train that largely disappears in val.
4. **Among the four models**, CatBoost and XGBoost are tied for lowest cost, the random forest is close behind, and **logistic regression is clearly worse** (it loses to the flat rule).
5. **The choice depends on the walk ceiling staying group-wide.** If the client required each hotel to stay under 10 walks per 1,000, the flat rule would be the cheapest method on val (Part 8).

**So the honest message for the client:** on the choose period, the best model is about as good as a well-tuned flat rule, not clearly better. The single test run will decide whether it saves anything. Do not promise savings before that (project rule 8).

## Setup

Loads train and val, the val probabilities from notebook 12, and the fitted preprocessor and models from notebooks 10 and 11. It defines two helpers used throughout:

- `night_tables[method]`: the contract's nightly table on val for each method;
- `apply_buffers(method, city, resort)`: scores the oversell rule `k = max(0, floor(expected) − buffer)` with a buffer for each hotel, using `contract.night_cost`.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "contract.py").exists():
        repo_root = folder
        break
if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import contract

split_dir = repo_root / "data" / "splits"
model_dir = repo_root / "data" / "models"
if not (model_dir / "val_results.csv").exists():
    raise FileNotFoundError("data/models/val_results.csv is missing. Run notebooks/12_validate_models.ipynb first.")

manifest = json.loads((split_dir / "split_columns.json").read_text(encoding="utf-8"))
inputs, label = manifest["inputs"], manifest["label"]


def load_split(name):
    # Test stays closed until every choice is frozen (project rule 2).
    if name not in ("train", "val"):
        raise ValueError("This notebook only opens train and val.")
    return pd.read_csv(split_dir / f"{name}.csv", index_col="row",
                       parse_dates=["arrival_date"], dtype={"agent": str, "company": str})


train, val = load_split("train"), load_split("val")
probabilities = pd.read_csv(model_dir / "val_probabilities.csv", index_col="row").loc[val.index]
methods = list(probabilities.columns)
models_only = [m for m in methods if m != "flat-rule reference"]
shared_buffers = json.loads((model_dir / "val_chosen_buffers.json").read_text(encoding="utf-8"))["chosen_buffers"]
val_scores = pd.read_csv(model_dir / "val_scores.csv", index_col="method")

night_tables = {m: contract.nightly_table(val[contract.REQUIRED], probabilities[m]) for m in methods}
HOTELS = ["City Hotel", "Resort Hotel"]


def oversell_counts(method, city, resort):
    table = night_tables[method]
    buffer = np.where(table.index.get_level_values("hotel") == "City Hotel", city, resort)
    return pd.Series(np.maximum(np.floor(table["expected"].to_numpy()) - buffer, 0), index=table.index)


def apply_buffers(method, city, resort):
    # Group result and each hotel's result for one method at one pair of buffers.
    table, k = night_tables[method], oversell_counts(method, city, resort)
    out = {"group": contract.night_cost(table, k)}
    for hotel in HOTELS:
        part = table.xs(hotel, level="hotel", drop_level=False)
        out[hotel] = contract.night_cost(part, k.loc[part.index])
    return out


print(f"val: {len(val):,} bookings, {len(night_tables[methods[0]])} hotel-nights; methods: {methods}")
print("Shared buffers chosen in notebook 12:", shared_buffers)

val: 29,546 bookings, 368 hotel-nights; methods: ['flat-rule reference', 'logistic regression', 'random forest', 'XGBoost', 'CatBoost']
Shared buffers chosen in notebook 12: {'flat-rule reference': 16, 'logistic regression': 6, 'random forest': 6, 'XGBoost': 5, 'CatBoost': 5}


## Part 2. What notebook 12 did, in four sentences

1. Each method gives every val booking a cancellation probability. `contract.nightly_table` adds them up per hotel-night into **expected cancellations**.
2. The oversell count is **k = max(0, ⌊expected⌋ − b)**: the expected count, rounded down, less a **buffer** *b* of safety rooms. This is the client's flat rule written as a formula, applied identically to every method (`docs/EVALUATION_CONTRACT.md`).
3. For each method, *b* is the cheapest whole number of rooms that keeps the **group** at or under **10 walks per 1,000 arrivals** on val.
4. Costs are then compared with the flat rule on the same 368 hotel-nights.

**Three facts you need for every answer below:**

- **An empty room costs the night's stay value** (mean `adr` × `nights`, often €300 to €600). **A walk costs €300.** So the cost-minimising rule wants to oversell aggressively, and **the walk ceiling is what stops it**. In notebook 12 every method's chosen buffer was exactly the smallest buffer that met the ceiling: **the ceiling is binding for every method.**
- **The buffer absorbs any constant bias.** If a method always expects 7 too few cancellations, it simply needs a smaller buffer. What the buffer **cannot** fix is how much the error **varies from night to night**. So comparing methods is really comparing how well each one tracks the ups and downs of cancellations.
- **The val costs are selection figures, not savings.** Each buffer was chosen on the same nights it is scored on.

## Part 3. Result 1: the models lost much of their training skill on val

**Evidence:** the table below (from notebooks 11 and 12).

In [2]:
display(val_scores[["ROC AUC (train)", "ROC AUC (val)", "log loss (train)", "log loss (val)",
                    "mean predicted rate (val)", "actual rate (val)"]]
        .style.format({"ROC AUC (train)": "{:.3f}", "ROC AUC (val)": "{:.3f}", "log loss (train)": "{:.3f}",
                       "log loss (val)": "{:.3f}", "mean predicted rate (val)": "{:.1%}", "actual rate (val)": "{:.1%}"}))

,ROC AUC (train),ROC AUC (val),log loss (train),log loss (val),mean predicted rate (val),actual rate (val)
method,,,,,,
flat-rule reference,0.582,0.557,0.638,0.652,36.4%,36.9%
logistic regression,0.817,0.705,0.462,0.611,28.1%,36.9%
random forest,0.810,0.701,0.480,0.601,29.3%,36.9%
XGBoost,0.815,0.714,0.458,0.606,27.8%,36.9%
CatBoost,0.815,0.702,0.459,0.606,27.9%,36.9%


**What it means.** On train, all four models ranked cancellations well (ROC AUC about 0.81–0.82). On val they fall to about **0.70–0.71**. Log loss worsens from about 0.46 to about 0.61. The flat rule barely changes, because it learnt almost nothing that could fail to carry over.

**The drop is not mainly overfitting.** Logistic regression, the least flexible model, falls just as far as XGBoost. Its 2,000-iteration fit on 143 columns cannot memorise 49,000 bookings. When every model falls by the same amount, the cause is in the **data**, not in any one model: the relationship between the inputs and cancellations changed between the periods. Part 4 shows what changed.

**Challenges.**
- *Isn't an AUC of 0.70 poor?* It is moderate. But the contract does not score AUC; it scores euros under the walk ceiling. AUC is a diagnostic. A model can rank well and still oversell badly if its probabilities are off, and the reverse.
- *Why report train scores at all?* The gap between train and val is the evidence that something shifted. Without it you could not tell a weak model from a changed world.
- *Could tuning fix it?* Tuning on val can improve the fit to val. But if the cause is a shift in one input (Part 4), tuning cannot recover a signal that is no longer there.

## Part 4. Result 2: every model expects too few cancellations, and why

**Evidence 1:** what happened to `previous_cancellations` and to the mix of bookings between train and val.

In [3]:
rows = []
for name, part in {"train": train, "val": val}.items():
    has_prior = part["previous_cancellations"].gt(0)
    rows.append({
        "period": name,
        "bookings with prior cancellations": f"{has_prior.sum():,} ({has_prior.mean():.1%})",
        "their cancellation rate": f"{part.loc[has_prior, label].mean():.1%}",
        "share of all cancellations they make up": f"{(part[label] & has_prior).sum() / part[label].sum():.1%}",
        "cancellation rate of everyone else": f"{part.loc[~has_prior, label].mean():.1%}",
        "overall cancellation rate": f"{part[label].mean():.1%}",
    })
display(pd.DataFrame(rows).set_index("period").T)

segment = pd.DataFrame({
    (name, measure): values
    for name, part in {"train": train, "val": val}.items()
    for measure, values in {"share of bookings": part["market_segment"].value_counts(normalize=True),
                            "cancellation rate": part.groupby("market_segment")[label].mean()}.items()
}).loc[["Online TA", "Offline TA/TO", "Groups", "Direct", "Corporate"]]
display(segment.style.format("{:.1%}"))

period,train,val
bookings with prior cancellations,"5,809 (11.8%)",397 (1.3%)
their cancellation rate,96.7%,70.3%
share of all cancellations they make up,31.9%,2.6%
cancellation rate of everyone else,27.6%,36.4%
overall cancellation rate,35.8%,36.9%


**Evidence 2:** how much each model relies on `previous_cancellations`. For the 5,809 train bookings with a prior cancellation, the cell compares each model's average probability with the count as recorded against the same bookings with the count set to 0. This measure is the **same for every model**. The importance scores in notebook 11 are **not** comparable across libraries: each library measures importance differently.

In [4]:
warnings.filterwarnings("ignore", message="Found unknown categories")
preprocessor = joblib.load(repo_root / "data" / "preprocessed" / "preprocessor.joblib")
with_prior = train[train["previous_cancellations"] > 0]
X_as_recorded = preprocessor.transform(with_prior[inputs]).to_numpy(dtype=float)
X_set_to_zero = preprocessor.transform(with_prior[inputs].assign(previous_cancellations=0)).to_numpy(dtype=float)

reliance = {}
for name in models_only:
    model = joblib.load(model_dir / f"{name.replace(' ', '_')}.joblib")
    reliance[name] = {"mean probability as recorded": model.predict_proba(X_as_recorded)[:, 1].mean(),
                      "mean probability with count set to 0": model.predict_proba(X_set_to_zero)[:, 1].mean()}
reliance = pd.DataFrame(reliance).T
reliance["drop"] = reliance.iloc[:, 0] - reliance.iloc[:, 1]
print(f"{len(with_prior):,} train bookings with a prior cancellation; {with_prior[label].mean():.1%} of them cancelled.")
display(reliance.style.format("{:.1%}"))

5,809 train bookings with a prior cancellation; 96.7% of them cancelled.


,mean probability as recorded,mean probability with count set to 0,drop
logistic regression,95.4%,38.3%,57.1%
random forest,84.5%,42.6%,41.9%
XGBoost,96.3%,37.6%,58.7%
CatBoost,96.1%,37.0%,59.1%


**What it means.**

- **In train, bookings with a prior cancellation are almost certain to cancel** (96.7%). They are 11.8% of train bookings but **about a third of all train cancellations**. Every model learnt this. Setting the count to 0 cuts their predicted probability from about 95% to under 40%.
- **In val, these bookings nearly vanish:** 1.3% of bookings, and only 70% of them cancel.
- **Meanwhile everyone else cancels more:** bookings with no prior cancellation go from 27.6% in train to **36.4%** in val. The models learnt "no prior cancellation means low risk", so they under-predict exactly these bookings.
- **The mix also moved towards online agencies:** Online TA rises from 36% to 54% of bookings, and its cancellation rate from 30% to 38%. Groups fall from 21% to 12%.

The flat rule uses only each hotel's train rate. That rate barely moved (City Hotel 41.7% → 40.8%, Resort Hotel 25.2% → 28.4%), so **the flat rule's level is about right**, while the models are about 9 points low.

**Challenges.**
- *Is this leakage?* It is consistent with the timing doubt recorded in notebook 06: `previous_cancellations` might count earlier bookings that were cancelled **later**, at the same moment as this booking. A burst of linked cancellations would look exactly like "prior cancellation → certain to cancel" in train. **But val cannot prove it** (notebook 07, P4). It can only show that the signal does not carry over. That is why the column is a **provisional** input.
- *Why would it vanish in val?* Two explanations fit, and the data cannot separate them. (1) A real change: fewer repeat customers with cancellation histories in late 2016. (2) An artefact of how the count is built: a booking's history can only include bookings created earlier, and if the large linked-cancellation blocks happened mostly in 2015–early 2016, train would hold most of them. Either way, a model should not lean on it.
- *Then why not just remove it?* That is a reasonable experiment to run on val before freezing (Part 11). It was not run here because notebook 12 compared the models as fitted, and any change of inputs is a new choice that must be compared on val like the others.
- *Is val an unusual period?* It is July to December only, so it has no spring. Test (January to August 2017) covers different months again. Seasonal differences are another reason to treat the val ranking as provisional.

## Part 5. Result 3: the shared buffer explains most of the 16% gap

**Evidence 1:** how each method's nightly error behaves at each hotel. **Bias** is the average of (expected − actual): below zero means too few cancellations expected. **Spread** is the standard deviation of that error from night to night. **Correlation** shows how well the expected count tracks the actual count across nights.

In [5]:
rows = []
for name in methods:
    table = night_tables[name]
    error = table["expected"] - table["cancelled"]
    for hotel in ["group", *HOTELS]:
        part = table if hotel == "group" else table.xs(hotel, level="hotel", drop_level=False)
        e = error.loc[part.index]
        rows.append({"hotel": hotel, "method": name, "bias (rooms)": e.mean(), "spread (rooms)": e.std(),
                     "correlation": np.corrcoef(part["expected"], part["cancelled"])[0, 1]})
display(pd.DataFrame(rows).set_index(["hotel", "method"]).style.format(
    {"bias (rooms)": "{:+.1f}", "spread (rooms)": "{:.1f}", "correlation": "{:.2f}"}))

print("Where the walks fall at the shared buffers chosen in notebook 12:")
display(pd.DataFrame({
    name: {f"{h}: walked guests": apply_buffers(name, b, b)[h]["walked_guests"] for h in HOTELS}
    | {f"{h}: empty rooms": apply_buffers(name, b, b)[h]["empty_rooms"] for h in HOTELS}
    for name, b in shared_buffers.items()
}).T)

,,bias (rooms),spread (rooms),correlation
hotel,method,,,
group,flat-rule reference,-0.3,14.0,0.88
City Hotel,flat-rule reference,+1.0,18.3,0.89
Resort Hotel,flat-rule reference,-1.7,7.2,0.73
group,logistic regression,-7.1,14.3,0.90
City Hotel,logistic regression,-11.1,18.1,0.88
Resort Hotel,logistic regression,-3.0,7.2,0.70
group,random forest,-6.1,13.8,0.92
City Hotel,random forest,-9.8,17.5,0.91
Resort Hotel,random forest,-2.3,6.9,0.72


Where the walks fall at the shared buffers chosen in notebook 12:


,City Hotel: walked guests,Resort Hotel: walked guests,City Hotel: empty rooms,Resort Hotel: empty rooms
flat-rule reference,167,2,3023,2509
logistic regression,117,56,3350,1748
random forest,116,64,3123,1649
XGBoost,103,53,3226,1610
CatBoost,106,53,3193,1607


**Evidence 2:** the same choice repeated with **a separate buffer for each hotel**: every pair of buffers from 0 to 60 rooms is tried, keeping only pairs that meet the **group** ceiling, and the cheapest pair is taken. Every method, including the flat rule, gets the same freedom.

In [6]:
BUFFERS = np.arange(0, 61)


def best_per_hotel_buffers(method, each_hotel_cap=None):
    # Cheapest (city, resort) pair meeting the group ceiling (and, optionally, a cap at each hotel).
    table, parts = night_tables[method], {}
    for hotel in HOTELS:
        t = table.xs(hotel, level="hotel")
        expected, actual, value = np.floor(t["expected"].to_numpy()), t["cancelled"].to_numpy(), t["stay_value"].to_numpy()
        k = np.maximum(expected[None, :] - BUFFERS[:, None], 0)
        empty, walks = np.clip(actual - k, 0, None), np.clip(k - actual, 0, None)
        parts[hotel] = {"cost": (empty * value).sum(1) + contract.WALK_COST * walks.sum(1), "walks": walks.sum(1),
                        "arrivals": (t["book"] - t["cancelled"]).sum()}
    city, resort = parts["City Hotel"], parts["Resort Hotel"]
    cost = city["cost"][:, None] + resort["cost"][None, :]
    group_rate = 1000 * (city["walks"][:, None] + resort["walks"][None, :]) / (city["arrivals"] + resort["arrivals"])
    allowed = group_rate <= contract.WALK_CEILING
    if each_hotel_cap is not None:
        allowed &= (1000 * city["walks"] / city["arrivals"] <= each_hotel_cap)[:, None]
        allowed &= (1000 * resort["walks"] / resort["arrivals"] <= each_hotel_cap)[None, :]
    if not allowed.any():
        return None
    i, j = np.unravel_index(np.argmin(np.where(allowed, cost, np.inf)), cost.shape)
    return int(BUFFERS[i]), int(BUFFERS[j])


def summarise(buffers):
    rows = []
    for name, (city, resort) in buffers.items():
        r = apply_buffers(name, city, resort)
        rows.append({"method": name, "City buffer": city, "Resort buffer": resort, "cost (EUR)": r["group"]["cost_eur"],
                     "empty rooms": r["group"]["empty_rooms"], "walked guests": r["group"]["walked_guests"],
                     "walks per 1,000 (group)": r["group"]["walks_per_1000"],
                     "walks per 1,000 (City)": r["City Hotel"]["walks_per_1000"],
                     "walks per 1,000 (Resort)": r["Resort Hotel"]["walks_per_1000"]})
    out = pd.DataFrame(rows).set_index("method")
    out["vs flat rule"] = out["cost (EUR)"] / out.loc["flat-rule reference", "cost (EUR)"] - 1
    return out


FMT = {"cost (EUR)": "{:,.0f}", "vs flat rule": "{:+.1%}", "walks per 1,000 (group)": "{:.2f}",
       "walks per 1,000 (City)": "{:.2f}", "walks per 1,000 (Resort)": "{:.2f}"}
shared = summarise({m: (b, b) for m, b in shared_buffers.items()})
per_hotel_buffers = {m: best_per_hotel_buffers(m) for m in methods}
per_hotel = summarise(per_hotel_buffers)

print("A. One shared buffer (notebook 12):")
display(shared.style.format(FMT))
print("B. A separate buffer for each hotel, group ceiling only:")
display(per_hotel.style.format(FMT))

A. One shared buffer (notebook 12):


,City buffer,Resort buffer,cost (EUR),empty rooms,walked guests,"walks per 1,000 (group)","walks per 1,000 (City)","walks per 1,000 (Resort)",vs flat rule
method,,,,,,,,,
flat-rule reference,16,16,"2,414,409",5532,169,9.06,13.96,0.30,+0.0%
logistic regression,6,6,"2,159,327",5098,173,9.28,9.78,8.37,-10.6%
random forest,6,6,"2,036,366",4772,180,9.65,9.70,9.57,-15.7%
XGBoost,5,5,"2,033,082",4836,156,8.36,8.61,7.92,-15.8%
CatBoost,5,5,"2,025,237",4800,159,8.52,8.86,7.92,-16.1%


B. A separate buffer for each hotel, group ceiling only:


,City buffer,Resort buffer,cost (EUR),empty rooms,walked guests,"walks per 1,000 (group)","walks per 1,000 (City)","walks per 1,000 (Resort)",vs flat rule
method,,,,,,,,,
flat-rule reference,20,4,"1,950,066",4870,182,9.76,8.44,12.11,+0.0%
logistic regression,5,6,"2,105,583",4929,186,9.97,10.87,8.37,+8.0%
random forest,7,5,"2,013,603",4791,180,9.65,8.36,11.96,+3.3%
XGBoost,6,3,"1,929,996",4695,174,9.33,7.44,12.71,-1.0%
CatBoost,6,3,"1,922,977",4660,178,9.54,7.61,13.01,-1.4%


**What it means.**

- **The models do not forecast the nights much better than the flat rule.** At the group level the spread of the nightly error is about the same for every method (about 14 rooms). Correlation with the actual count is only slightly higher for the models (0.90–0.92 against 0.88). At the Resort Hotel the spreads are 6.8 to 7.2 rooms for everyone.
- **But their bias differs sharply between the hotels.** The models expect about **11** too few cancellations per night at the City Hotel and about **3** too few at the Resort. The flat rule is close to zero at both. Under-predicting acts like extra safety. **The models' bias quietly gives the City Hotel more protection than the Resort**, which is what a revenue manager would want.
- **With one shared buffer, the flat rule cannot do that.** Its walks are almost all at the City Hotel (167 of 169). So it needs **16 rooms** of safety everywhere, which is far too many at the Resort: only 2 walks there, but 2,509 empty rooms.
- **Give every method a buffer per hotel (table B)** and the flat rule drops from €2.41M to **€1.95M**. It chooses 20 rooms for the City Hotel and 4 for the Resort. The models improve too, but much less, because their bias was already doing that job. The result: **CatBoost −1.4%, XGBoost −1.0%, random forest +3.3%, logistic regression +8.0%** against the flat rule.

So most of the 16% in notebook 12 measured **one buffer for two very different hotels**, not better forecasting.

**Challenges.**
- *Is a per-hotel buffer allowed?* The contract describes the flat rule as "a fixed number of rooms for safety" and does not say whether it may differ by hotel. Whether the client's margin differs by hotel is an **open client question** (`docs/DECISIONS.md`, `docs/CHARTER.md`). A per-hotel buffer is a setting chosen on the choose period, exactly like the shared one. The key point is fairness: **whatever structure the model gets, the flat rule must get too.**
- *Isn't table A then wrong?* It is correct for a shared buffer. It just answers a narrower question, and it hands the models an advantage the client's own rule could get without any model.
- *Why does the walk ceiling still hold if the Resort Hotel walks 12–13 per 1,000 in table B?* The ceiling is **group-wide** (project rule 6), and the group stays under 10. The client has given **no per-hotel threshold**, so this is reported, not judged. It is a fair question for the client (Part 8).
- *Why do the models expect too few cancellations at the City Hotel in particular?* In train, the City Hotel held 84% of the bookings with a prior cancellation and 71% of Groups bookings, against 64% of all bookings. Those are exactly the signals that weakened in val (Part 4), so the City Hotel loses the most.

## Part 6. Result 4: how sure can we be?

Two checks, both using **the buffers already chosen** (table B, and table A for comparison):

1. **Month by month:** does a method win steadily, or in a few months only?
2. **Resampling weeks:** draw the 27 val weeks at random with replacement 4,000 times, and count how often each model costs less than the flat rule. Weeks are resampled rather than nights because neighbouring nights are alike (same season, same events). Resampling single nights would overstate certainty.

This is **generous to every method**: it keeps each buffer fixed, so it ignores that the buffers were themselves chosen on these weeks.

In [7]:
def nightly_costs(method, city, resort):
    table, k = night_tables[method], oversell_counts(method, city, resort)
    return ((table["cancelled"] - k).clip(lower=0) * table["stay_value"]
            + (k - table["cancelled"]).clip(lower=0) * contract.WALK_COST)


def by_month_and_bootstrap(buffers, title):
    costs = {m: nightly_costs(m, *buffers[m]) for m in methods}
    dates = night_tables[methods[0]].index.get_level_values("arrival_date")
    monthly = pd.DataFrame({m: c.groupby(dates.month).sum() for m, c in costs.items()})
    monthly.index = [pd.Timestamp(2016, mo, 1).strftime("%b %Y") for mo in monthly.index]
    weekly = pd.DataFrame({m: c.groupby(dates.to_period("W")).sum() for m, c in costs.items()})
    draws = np.random.default_rng(0).integers(0, len(weekly), (4000, len(weekly)))
    totals = {m: weekly[m].to_numpy()[draws].sum(axis=1) for m in methods}
    summary = pd.DataFrame({
        m: {"months cheaper than flat rule": f"{int((monthly[m] < monthly['flat-rule reference']).sum())} of {len(monthly)}",
            "resamples cheaper than flat rule": f"{(totals[m] < totals['flat-rule reference']).mean():.0%}",
            "difference vs flat rule, 90% range (EUR)":
                "{:+,.0f} to {:+,.0f}".format(*np.percentile(totals[m] - totals["flat-rule reference"], [5, 95]))}
        for m in models_only
    }).T
    print(title)
    display((monthly / 1000).round(0).astype(int).rename(columns=lambda c: c + " (EUR k)"))
    display(summary)
    return totals


totals_shared = by_month_and_bootstrap({m: (b, b) for m, b in shared_buffers.items()}, "A. One shared buffer:")
totals_per_hotel = by_month_and_bootstrap(per_hotel_buffers, "B. A separate buffer for each hotel:")
cat_vs_xgb = (totals_per_hotel["CatBoost"] < totals_per_hotel["XGBoost"]).mean()
print(f"B: CatBoost cheaper than XGBoost in {cat_vs_xgb:.0%} of resamples.")

A. One shared buffer:


,flat-rule reference (EUR k),logistic regression (EUR k),random forest (EUR k),XGBoost (EUR k),CatBoost (EUR k)
Jul 2016,483,424,398,389,393
Aug 2016,663,577,536,528,529
Sep 2016,404,352,331,335,333
Oct 2016,384,378,361,362,361
Nov 2016,228,194,190,197,190
Dec 2016,252,234,221,222,219


,months cheaper than flat rule,resamples cheaper than flat rule,"difference vs flat rule, 90% range (EUR)"
logistic regression,6 of 6,100%,"-349,664 to -157,587"
random forest,6 of 6,100%,"-479,467 to -275,439"
XGBoost,6 of 6,100%,"-490,557 to -271,129"
CatBoost,6 of 6,100%,"-495,463 to -282,690"


B. A separate buffer for each hotel:


,flat-rule reference (EUR k),logistic regression (EUR k),random forest (EUR k),XGBoost (EUR k),CatBoost (EUR k)
Jul 2016,334,415,382,348,351
Aug 2016,455,565,519,482,483
Sep 2016,333,346,329,319,318
Oct 2016,355,369,364,359,358
Nov 2016,229,186,195,201,194
Dec 2016,245,225,225,221,219


,months cheaper than flat rule,resamples cheaper than flat rule,"difference vs flat rule, 90% range (EUR)"
logistic regression,2 of 6,1%,"+40,330 to +271,418"
random forest,3 of 6,10%,"-16,398 to +143,154"
XGBoost,3 of 6,69%,"-83,770 to +41,865"
CatBoost,3 of 6,74%,"-94,500 to +36,652"


B: CatBoost cheaper than XGBoost in 91% of resamples.


**What it means.**

- **With a shared buffer (A)**, every model beats the flat rule in every month and in essentially every resample. That looks decisive, but Part 5 showed it is mostly the buffer structure.
- **With per-hotel buffers (B)**, the best models (CatBoost, XGBoost) win in only **3 of 6 months** and in about **70–75%** of resamples. Their 90% range runs from a clear saving to a clear loss. **That is not a reliable difference.** The random forest wins in about 10% of resamples, and logistic regression almost never.
- **The months split by season.** In B the models lose in July, August and October and win in September, November and December. A model that wins only in part of the year may not win in January to August 2017 (test).

**Challenges.**
- *74% sounds good. Why not call it a win?* By convention you would want around 95% before calling a difference reliable. And this check is generous: the buffers were chosen on these same weeks. 74% means "slightly more likely than not".
- *Why resample weeks rather than nights?* Nights next to each other share season, events and group blocks. Treating them as independent would make the result look more certain than it is.
- *Isn't 27 weeks too few?* It is what the choose period holds. That is exactly why the result is uncertain and why test, run once, is the real verdict.

## Part 7. Result 5: choosing among the four models

| Question | Logistic regression | Random forest | XGBoost | CatBoost |
|---|---|---|---|---|
| Cost under a shared buffer (A) | 4th | 3rd | 2nd | **1st** |
| Cost under per-hotel buffers (B) | 4th, worse than flat rule | 3rd, worse than flat rule | 2nd, −1.0% | **1st, −1.4%** |
| Beats the flat rule reliably in B? | No | No | No (≈70%) | No (≈75%) |
| ROC AUC on val | 0.705 | 0.701 | **0.714** | 0.702 |
| Reliance on `previous_cancellations` (Part 4) | high | somewhat lower | high | high |
| Walk headroom under the ceiling in A | 9.28 | 9.65 (tightest) | **8.36** | 8.52 |

**Reasoning.**

1. **The project's selection rule is cost under the walk ceiling on the choose period** (PRD, `docs/DECISIONS.md`). It was set **before** any result, so it should decide unless there is a strong reason not to. On that rule **CatBoost is first under both buffer structures**, and it is cheaper than XGBoost in about 90% of resamples (B).
2. **The margin over XGBoost is small** (about €7,000 on €1.9M, under 0.5%). XGBoost has slightly higher AUC and slightly more walk headroom. Neither difference is large enough to override the pre-set rule. **Choosing XGBoost would be equally defensible**, provided you say why before seeing test.
3. **Logistic regression is out:** it is clearly the most expensive and loses to the flat rule in B in almost every resample.
4. **The random forest is out:** it costs more than the flat rule in B, and it runs closest to the walk ceiling in A (9.65).
5. **Do not choose between XGBoost and CatBoost on reliance on `previous_cancellations`.** Measured the same way for both (Part 4), they rely on it equally. The very different importance percentages in notebook 11 come from how each library measures importance, not from real behaviour.

**Challenges.**
- *Why not the model with the best AUC (XGBoost)?* The contract scores euros under the ceiling, not AUC. AUC measures ranking, while the decision uses the **sum** of probabilities per night.
- *CatBoost is a "black box". Shouldn't the client get something simpler?* The simplest candidate, logistic regression, costs clearly more. The PRD's simpler fallback, a hotel-by-history lookup, was planned but **not built**. If the client values simplicity above a 1% difference, the per-hotel flat rule itself is the simplest option, and on val it is about as good.

## Part 8. The two hotels separately

**Evidence:** each hotel's results at the per-hotel buffers (table B), and what changes if the client were to require **each** hotel to stay at or under 10 walks per 1,000. That is a **hypothetical**: the client has stated only a group-wide ceiling, so it is not applied as a requirement (project rule 6).

In [8]:
rows = []
for name, (city, resort) in per_hotel_buffers.items():
    r = apply_buffers(name, city, resort)
    for hotel in HOTELS:
        rows.append({"hotel": hotel, "method": name, "buffer": city if hotel == "City Hotel" else resort,
                     "cost (EUR)": r[hotel]["cost_eur"], "empty rooms": r[hotel]["empty_rooms"],
                     "walked guests": r[hotel]["walked_guests"], "walks per 1,000": r[hotel]["walks_per_1000"]})
display(pd.DataFrame(rows).set_index(["hotel", "method"]).sort_index().style.format({"cost (EUR)": "{:,.0f}", "walks per 1,000": "{:.2f}"}))

capped_buffers = {m: best_per_hotel_buffers(m, each_hotel_cap=contract.WALK_CEILING) for m in methods}
print("Hypothetical: each hotel also at or under 10 walks per 1,000 (not a client requirement):")
display(summarise(capped_buffers).style.format(FMT))

Hypothetical: each hotel also at or under 10 walks per 1,000 (not a client requirement):


,City buffer,Resort buffer,cost (EUR),empty rooms,walked guests,"walks per 1,000 (group)","walks per 1,000 (City)","walks per 1,000 (Resort)",vs flat rule
method,,,,,,,,,
flat-rule reference,19,5,"1,977,485",4869,175,9.38,9.53,9.12,+0.0%
logistic regression,6,6,"2,159,327",5098,173,9.28,9.78,8.37,+9.2%
random forest,6,6,"2,036,366",4772,180,9.65,9.70,9.57,+3.0%
XGBoost,5,5,"2,033,082",4836,156,8.36,8.61,7.92,+2.8%
CatBoost,5,5,"2,025,237",4800,159,8.52,8.86,7.92,+2.4%


**What it means.**

- **The City Hotel is the hard one:** more bookings, larger group blocks, and the largest nightly errors (spread of about 18 rooms against about 7 at the Resort). Every method's walks are hardest to control there.
- **With per-hotel buffers and only a group ceiling, every method except logistic regression lets the Resort walk more than 10 per 1,000** (about 12–13; CatBoost 13.0), balanced by fewer at the City Hotel. That is allowed under the group-wide ceiling, but a client might not like it at one property.
- **If the client did require each hotel to stay under 10** (the hypothetical table), **the ranking reverses: the flat rule becomes the cheapest method.** It moves to buffers of 19 (City) and 5 (Resort), and every model costs more than it (CatBoost +2.4%, XGBoost +2.8%, random forest +3.0%, logistic regression +9.2%). The models end up back at their shared buffers, because those already keep both hotels under 10.

So the case for a model **depends on the client accepting the group-wide ceiling as the only limit**. That is what the brief says today, but it is worth confirming before test.

**Challenges.**
- *Is it acceptable that the Resort walks 13 per 1,000?* The brief states one group-wide brand standard, so yes as stated. It is a reasonable question to put to the client before test, not after, because the answer decides whether a model or the flat rule is preferred.
- *Shouldn't you report the hotels separately on test?* Yes, always (project rule 6), without claiming a per-hotel threshold the client did not give.

## Part 9. The recommendation, and what you may claim

**Freeze before test:**

| Setting | Value | Chosen how |
|---|---|---|
| Method | **CatBoost** (as fitted in notebook 11, on the prepared inputs from notebook 10) | Lowest choose-period cost under the walk ceiling, under both buffer structures |
| Buffer structure | **A separate buffer per hotel**, the same structure for the flat rule | Fair comparison: the flat rule gets the same freedom |
| Buffers | **City Hotel 6, Resort Hotel 3** | Cheapest pair meeting the group ceiling on val |
| Flat-rule benchmark | Train hotel rates (City 41.7%, Resort 25.2%), **City 20, Resort 4** | Same selection, same ceiling |
| Rule | k = max(0, ⌊expected⌋ − buffer) | Same for every method |

Record these in `docs/DECISIONS.md` **before** opening test, with the reasons from Parts 5 to 7.

**You may say:**
- On the choose period, with each method's buffer set the same way, the best model (CatBoost) cost about **1.4% less** than the flat rule. The difference **is not reliable**: it was cheaper in about three quarters of resampled weeks, and in 3 of 6 months.
- All methods met the group walk ceiling on the choose period.
- The models expect too few cancellations on the choose period, mainly because the signal in `previous_cancellations` did not carry over from train.

**You may not say (project rule 8):**
- That the model **saves** money: the choose-period figures were used to choose; only test can show a saving.
- That it saves **16%**: that figure depends on giving the flat rule a single shared buffer.
- That the model is a **live morning forecast**: the data is the final book, not a morning snapshot (project rule 7).
- That it **validates** the vendor's accuracy claim: nothing here tests that.

**Two decisions before freezing (yours, not the notebook's):**
1. **Shared or per-hotel buffers.** This notebook recommends per-hotel, applied to both. If the team keeps a shared buffer, report both versions, because the shared version flatters the model.
2. **Whether to first test the models without `previous_cancellations`** (or without all four provisional inputs) on val. It is allowed, and it may fix the under-prediction. Any variant must be compared with the flat rule exactly as above, and the decision recorded before test.

## Part 10. The hardest questions

**"Notebook 12 said the models save 16%. Now you say 1.4%. Which is it?"**
Both are correct answers to different questions. With one shared buffer the gap is about 16%. But the flat rule is badly handicapped there: it must protect the City Hotel with a buffer that wastes rooms at the Resort. Give both the same per-hotel freedom and the gap is about 1.4%, and not reliable. The second is the fair comparison. Neither is a savings figure: both are on the period used to choose.

**"If the models barely beat a flat rule, what was the point?"**
Finding out is the point. The client's question is whether a model beats their rule. On the choose period the honest answer is "about equal". The main lesson is that a per-hotel buffer, which needs no model, captures most of the improvement. That is valuable to the client on its own.

**"Your models under-predict cancellations by 9 points. Isn't that a broken model?"**
The level is wrong, mainly because one input's signal did not carry over (Part 4). The buffer absorbs a steady bias, which is why the chosen buffers are small (3 to 6 rooms). What the buffer cannot absorb is night-to-night error, and there the models are about as good as the flat rule. A bias that differs between periods is still a risk for test, because the right buffer could change.

**"Did you look at the test period?"**
No. No notebook from 08 onwards reads test outcomes; the loaders refuse to open test. Everything was chosen on train and val.

**"Why should I believe the leakage checks?"**
Notebook 10 checked preprocessing twice: fit on train only, no label used, rows transformed independently, no identifiers. Notebook 12 checked that no model or preprocessor changed while predicting val. The one thing no check can settle is **timing** (whether each input was known on the forecast morning). Four inputs are provisional for that reason, and Part 4 shows one of them behaving suspiciously.

**"Why not tune the models to do better?"**
Tuning on val is allowed, but every extra choice made on val makes val a less honest guide. That is why the gains on val come out inflated. The models are close to the flat rule because of a data shift, not a lack of tuning. Tuning cannot recover a signal that disappeared.

**"Why CatBoost and not XGBoost?"**
The project's pre-set criterion is lowest choose-period cost under the ceiling, and CatBoost leads under both buffer structures. The margin is tiny, and XGBoost would be equally defensible. What matters is to fix the choice and its reason **before** test.

**"What will happen on test?"**
Unknown, and you should say so. Test covers January to August 2017, which has spring and summer months and none of val's autumn months. The models lost to the flat rule in July and August on val. Expect the result to be close to the flat rule in either direction, and report it honestly whichever way it falls.

## Part 11. What would change the recommendation

| New evidence | What changes |
|---|---|
| The client says the flat-rule margin **must** be one shared number | Use table A. Report it with the caveat that the gap mostly reflects the buffer structure. |
| The client sets a **per-hotel** walk threshold of 10 | On val the **flat rule becomes the best method** (Part 8): recommend it, or a model only if a revised one beats it under that limit. |
| Models refitted **without `previous_cancellations`** beat the per-hotel flat rule more clearly on val | Prefer that variant. It is also less exposed to the provisional timing doubt. |
| The **hotel-by-history lookup** from the PRD is built and matches CatBoost on val | Prefer the lookup: the PRD prefers the simpler method when results are indistinguishable. |
| A change-log check shows `previous_cancellations` counts later cancellations | Drop it (and anything built on it) and refit; every current model result becomes invalid. |
| Test shows no saving | Report it as found. "A per-hotel buffer on the flat rule is as good as the models" is a legitimate, useful result. |

## Final cell: are the numbers quoted above still current?

The explanations quote specific numbers. This cell recomputes them and lists any that no longer match, for example after notebooks 06 to 12 are rerun with different choices. Re-read and rewrite those parts if anything is listed.

In [9]:
quoted = {
    "shared buffers": (shared_buffers, {"flat-rule reference": 16, "logistic regression": 6, "random forest": 6,
                                        "XGBoost": 5, "CatBoost": 5}),
    "per-hotel buffers": (per_hotel_buffers, {"flat-rule reference": (20, 4), "logistic regression": (5, 6),
                                              "random forest": (7, 5), "XGBoost": (6, 3), "CatBoost": (6, 3)}),
    "per-hotel flat-rule cost (EUR k)": (round(per_hotel.loc["flat-rule reference", "cost (EUR)"] / 1000), 1950),
    "shared flat-rule cost (EUR k)": (round(shared.loc["flat-rule reference", "cost (EUR)"] / 1000), 2414),
    "CatBoost vs flat, per-hotel (%)": (round(100 * per_hotel.loc["CatBoost", "vs flat rule"], 1), -1.4),
    "XGBoost vs flat, per-hotel (%)": (round(100 * per_hotel.loc["XGBoost", "vs flat rule"], 1), -1.0),
    "random forest vs flat, per-hotel (%)": (round(100 * per_hotel.loc["random forest", "vs flat rule"], 1), 3.3),
    "logistic regression vs flat, per-hotel (%)": (round(100 * per_hotel.loc["logistic regression", "vs flat rule"], 1), 8.0),
    "CatBoost resamples cheaper than flat, per-hotel (%)":
        (round(100 * (totals_per_hotel["CatBoost"] < totals_per_hotel["flat-rule reference"]).mean()), 74),
    "XGBoost resamples cheaper than flat, per-hotel (%)":
        (round(100 * (totals_per_hotel["XGBoost"] < totals_per_hotel["flat-rule reference"]).mean()), 69),
    "lowest-cost model, per-hotel": (per_hotel.drop("flat-rule reference")["cost (EUR)"].idxmin(), "CatBoost"),
    "lowest-cost model, shared": (shared.drop("flat-rule reference")["cost (EUR)"].idxmin(), "CatBoost"),
    "cheapest method with each hotel capped at 10 (hypothetical)":
        (summarise(capped_buffers)["cost (EUR)"].idxmin(), "flat-rule reference"),
    "CatBoost vs flat with each hotel capped (%)": (round(100 * summarise(capped_buffers).loc["CatBoost", "vs flat rule"], 1), 2.4),
    "City Hotel share of train prior-cancellation bookings (%)":
        (round(100 * train.loc[train["previous_cancellations"] > 0, "hotel"].eq("City Hotel").mean()), 84),
    "val actual cancellation rate (%)": (round(100 * val[label].mean(), 1), 36.9),
    "train share with prior cancellations (%)": (round(100 * train["previous_cancellations"].gt(0).mean(), 1), 11.8),
    "val share with prior cancellations (%)": (round(100 * val["previous_cancellations"].gt(0).mean(), 1), 1.3),
}
stale = [f"{name}: text says {expected}, data now gives {actual}" for name, (actual, expected) in quoted.items()
         if actual != expected]
if stale:
    print(f"{len(stale)} quoted number(s) out of date; revise the text:")
    for item in stale:
        print("  -", item)
else:
    print(f"All {len(quoted)} quoted numbers match the data.")

All 18 quoted numbers match the data.
